# Step 1: Data preprocessing (JDSearch)

Cleans the raw search logs and product catalogue, splits slates into train/val1/val2/test, and negative-samples the training slates.

**Before running:** add a shortcut to the shared `Grp8Submission` folder in your My Drive, with the raw files `user_behavior_data.txt` and `product_meta_data.txt` in its `raw_data/` folder.

**Outputs** `step1/` folder will include `train_sampled.parquet`, `val1.parquet`, `val2.parquet`, `test.parquet`, `product_metadata_clean.parquet` after this file has been run.

**Run order:** first notebook. Restart the runtime, then run all cells top to bottom.

In [1]:
# import all libraries
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow
import pyarrow.compute
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# Setup: mount Google Drive (same project folder in every notebook)
from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
RAW_DIR = PROJECT / "raw_data"  
DATA = PROJECT / "step1"     
DATA_DIR = DATA

USER_FILE = DATA_DIR / "user_behavior_data.txt"
PRODUCT_FILE = DATA_DIR / "product_meta_data.txt"
USER_RAW = DATA_DIR / "user_behavior_raw.parquet"
PRODUCT_RAW = DATA_DIR / "product_metadata_raw.parquet"

# Convert the raw .txt files to parquet (first run only; later runs load the parquet, which is faster)
if not USER_RAW.exists():
    pd.read_csv(USER_FILE, sep="\t", header=0, dtype=str, na_filter=False).to_parquet(USER_RAW, index=False)
if not PRODUCT_RAW.exists():
    pd.read_csv(PRODUCT_FILE, sep="\t", header=0, dtype=str, na_filter=False).to_parquet(PRODUCT_RAW, index=False)

Mounted at /content/drive


### Step 1a: Data preprocessing

- **Step 1: Load** the raw search logs and product catalogue (cached as parquet for faster reruns).
- **Step 2: Assign `slate_id`**, one ID per search, used to join every later file.
- **Step 3: Parse separator** The raw file stores each list as one string with the `_` separator for candidates, labels and history, while the character `\x18` as a separator for the words of a query or product name. Splitting gives proper lists.
- **Step 4: Validate values.** Labels must be 0–3 and action types CLICK, CART, FLW or ORD. Time gaps must be integers.
- **Step 5: Fix data types.** The raw file is read as text, so labels look like `"1.0"` and product IDs like `"101"` by converting them into integers, we can avoid format mismatch and do arithmetic calculations on them. 
- **Step 6: Check list lengths.** Each search stores its data as lists paired by position so this is a validation check to ensure the lists that are paired are of the same length.
- **Step 7: Flag query-less history actions (`has_query`).** A query of `-1` means the past action did not come from a search.
- **Step 8: Compute days before the search.** The raw file stores the time gap, in seconds, between one action and the next but by summing them from the end allows us to find out each action's age at search time.
- **Step 9: Flag missing metadata (`has_metadata`).** About 0.45% of candidates have no catalogue entry. To prevent any errors due to the lack of data, we have included this so that later steps can treat their features as missing rather than zero.
- **Step 10: Shuffle candidates within each search.** Removes the original display order, so no model can learn from a product's position in the list.
- **Step 11: Save the cleaned product metadata.** The cleaned search data is saved per split in Step 1c.

In [3]:
# Step 1: run to load the parquet files
user_df = pd.read_parquet(USER_RAW)
product_df = pd.read_parquet(PRODUCT_RAW)

In [4]:
# Step 2: assign a unique slate_id to each row
user_df.insert(0, "slate_id", np.arange(len(user_df), dtype=np.int32))

In [ ]:
# Step 3: parse separators
def split_lists(df, cols, sep="_"):
    # Split each column's text into a list: "101_202_303" -> ["101", "202", "303"]
    for col in cols:
        df[col] = df[col].astype("large_string[pyarrow]").str.split(sep)
    return df

LIST_COLS = ["candidate_wid_list", "candidate_label_list",
             "history_qry_list", "history_wid_list",
             "history_type_list", "history_time_list"]

user_df = split_lists(user_df, LIST_COLS)
user_df = split_lists(user_df, ["query"], sep="\x18")
product_df = split_lists(product_df, ["name"], sep="\x18")

# each past action's query "36547097\x18602574" -> ["36547097", "602574"]
qry_arr = pyarrow.array(user_df["history_qry_list"])
qry_terms = pyarrow.compute.split_pattern(qry_arr.flatten(), "\x18")
qry_nested = pyarrow.ListArray.from_arrays(
    pyarrow.compute.subtract(qry_arr.offsets, qry_arr.offsets[0]), qry_terms)
user_df["history_qry_list_terms"] = pd.Series(
    qry_nested, index=user_df.index, dtype=pd.ArrowDtype(qry_nested.type))

In [6]:
# Step 4: validate allowed values
labels = user_df["candidate_label_list"].list.flatten()
types = user_df["history_type_list"].list.flatten()
times = user_df["history_time_list"].list.flatten()

print("Labels:", labels.unique())
print("Types:", types.unique())
print("Invalid time gaps:", (~times.str.isdigit()).sum())

Labels: <ArrowExtensionArray>
['1.0', '0.0', '2.0', '3.0']
Length: 4, dtype: large_string[pyarrow]
Types: <ArrowExtensionArray>
['ORD', 'CART', 'CLICK', 'FLW']
Length: 4, dtype: large_string[pyarrow]
Invalid time gaps: 0


In [7]:
# Step 5: convert text/float to integers
def to_int(df, cols, is_list=True):
    # Convert columns from text to integers. Lists of text -> lists of integers.
    for col in cols:
        if is_list:
            df[col] = (df[col]
                       .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.float64())))
                       .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.int64()))))
        else:
            df[col] = df[col].astype("int64")
    return df

user_df = to_int(user_df, ["candidate_wid_list", "candidate_label_list",
                           "history_wid_list", "history_time_list"])

product_df = to_int(product_df, ["wid", "brand_id", "cate_id_1", "cate_id_2",
                                 "cate_id_3", "cate_id_4", "shop_id"], is_list=False)

In [ ]:
# Step 6: check list lengths match within each row
cand_len = user_df["candidate_wid_list"].list.len()
label_len = user_df["candidate_label_list"].list.len()
print("Candidate/label mismatches:", (cand_len != label_len).sum())   

hist_len = user_df["history_wid_list"].list.len()
for col in ["history_qry_list", "history_type_list"]:
    print(col, "mismatches:", (user_df[col].list.len() != hist_len).sum())   

# history_time_list has one extra value: the gap to the current search (first value is always 0)
print("history_time_list mismatches:",
      (user_df["history_time_list"].list.len() != hist_len + 1).sum())   

Candidate/label mismatches: 0
history_qry_list mismatches: 0
history_type_list mismatches: 0
history_time_list mismatches: 0


In [9]:
# Step 7: flag whether each past action came from a search ("-1" = no query)

def add_has_query(df):
    # True/False per past action: did it come from a search?
    arr = pyarrow.array(df["history_qry_list"])
    flags = pyarrow.compute.not_equal(arr.flatten(), "-1")
    out = pyarrow.ListArray.from_arrays(pyarrow.compute.subtract(arr.offsets, arr.offsets[0]), flags)
    df["history_has_query"] = pd.Series(out, index=df.index, dtype=pd.ArrowDtype(out.type))
    return df

user_df = add_has_query(user_df)

In [ ]:
# Step 8: compute days before the search
def days_before(times):
    t = np.asarray(times)
    return np.cumsum(t[::-1])[::-1][1:] / 86400

user_df["history_days_before"] = (user_df["history_time_list"]
                                  .apply(days_before)
                                  .astype(pd.ArrowDtype(pyarrow.list_(pyarrow.float64()))))

In [11]:
# Step 9: add has_metadata flags
def add_has_metadata(df, wid_col, new_col, known_wids):
    # True/False per product in a list column: is it in the product metadata?
    arr = pyarrow.array(df[wid_col])
    flags = pyarrow.array(pd.Series(arr.flatten()).isin(known_wids).to_numpy())
    out = pyarrow.ListArray.from_arrays(pyarrow.compute.subtract(arr.offsets, arr.offsets[0]), flags)
    df[new_col] = pd.Series(out, index=df.index, dtype=pd.ArrowDtype(out.type))
    return df

user_df = add_has_metadata(user_df, "candidate_wid_list", "candidate_has_metadata", product_df["wid"])
user_df = add_has_metadata(user_df, "history_wid_list", "history_has_metadata", product_df["wid"])

In [12]:
# Step 10: shuffle candidates within each slate
SEED = 42

def shuffle_slates(df, cols, seed=SEED):
    # Shuffle each row's candidates, moving all listed columns in the same order.
    rng = np.random.default_rng(seed)
    perms = [rng.permutation(n) for n in df[cols[0]].list.len()]
    for col in cols:
        dtype = df[col].dtype
        df[col] = pd.Series([np.asarray(x)[p] for x, p in zip(df[col], perms)], index=df.index).astype(dtype)
    return df

user_df = shuffle_slates(user_df, ["candidate_wid_list", "candidate_label_list", "candidate_has_metadata"])

In [13]:
print(user_df.head())

   slate_id                                              query  \
0         0      ['63995226' '60257419' '61002446' '25575117']   
1         1                            ['12549634' '27438884']   
2         2                  ['2437437' '60677689' '39206305']   
3         3  ['29463948' '63209534' '93568451' '64880503' '...   
4         4                   ['2793914' '96436578' '3663289']   

                                  candidate_wid_list  \
0  [97857938 38739023 60065049 34921664 73938759 ...   
1  [37849772 10338877 33382654 30308565 15104374 ...   
2  [42155506 18246636 21020315 85184167 83306805 ...   
3  [14732314 31265387 53977576 90043344 64739295 ...   
4  [91983072 79730178  1792849 14397314 96966404 ...   

                                candidate_label_list  \
0  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0...   
1  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0...   
2  [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0...   
3                        [0 0 0 0 0 0 0 0 

In [14]:
PRODUCT_COLS = ["wid", "name", "brand_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "shop_id"]
product_df[PRODUCT_COLS].to_parquet(DATA_DIR / "product_metadata_clean.parquet", index=False)

### Step 1b: Train/Val1/Val2/Test Splitting

Stratified, grouped 70/10/10/10 split
- Stratified on: `has_purchase`, history-length bucket, slate-length bucket
- Grouped on: `query` string (rows sharing a query always land in the same split)
- Done by cutting the data into 10 folds: 7 -> train, 1 -> val1, 1 -> val2, 1 -> test

**Role of each split**
- `train`: fits the models (negative-sampled in Step 1c).
- `val1`: early stopping and hyperparameter tuning. For the short-term model (Model 1, LightGBM), the chosen configuration is the one with the best holdout NDCG@10 on `val1`.
- `val2`: used only on the final models, after tuning is done. It calibrates P1 and P2 with Platt scaling (fitted separately for each model), then fits the gate.
- `test`: final, untouched evaluation.

Input to `split_data`: cleaned slate-level dataframe (one row per search/user).
Required columns:
`slate_id`, `query` (string), `candidate_wid_list`, `candidate_label_list` (arrays of equal length), `history_wid_list` (array)

In [15]:
# split into train/val1/val2/test
SEED = 42

def add_split_helper_cols(df, hist_bins=(0, 29, 78, 188, np.inf), slate_bins=(0, 76, 90, 95, np.inf)):
    # Adds has_purchase, hist_len, slate_len, query_key and the combined stratum label
    out = df.copy()
    out["query_key"] = out["query"].apply(" ".join)   # ["12", "845"] -> "12 845", so it can be grouped
    out["has_purchase"] = out["candidate_label_list"].apply(lambda x: int((np.asarray(x) == 3).any()))
    out["hist_len"] = out["history_wid_list"].apply(len)
    out["slate_len"] = out["candidate_wid_list"].apply(len)
    hist_b = pd.cut(out["hist_len"], bins=list(hist_bins), labels=False, include_lowest=True)
    slate_b = pd.cut(out["slate_len"], bins=list(slate_bins), labels=False, include_lowest=True)
    out["stratum"] = (out["has_purchase"].astype(str) + "_" + hist_b.astype(str) + "_" + slate_b.astype(str))
    return out


def split_data(df, seed=SEED, n_folds=10, n_val1_folds=1, n_val2_folds=1, n_test_folds=1):
    # 70/10/10/10 with the defaults. Rows with the same query stay in the same split.
    out = add_split_helper_cols(df)
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    fold = np.full(len(out), -1, dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(out, y=out["stratum"], groups=out["query_key"])):
        fold[idx] = k
    assert (fold >= 0).all()

    n_train = n_folds - n_val1_folds - n_val2_folds - n_test_folds
    split = np.select(
        [fold < n_train,
         fold < n_train + n_val1_folds,
         fold < n_train + n_val1_folds + n_val2_folds],
        ["train", "val1", "val2"],
        default="test",
    )
    out["split"] = split
    return out


def check_split(df):
    # Prints proportions and sanity checks.
    print("Rows per split:\n", df["split"].value_counts(normalize=True).round(4), "\n")
    summary = df.groupby("split").agg(
        rows=("slate_id", "size"),
        purchase_rate=("has_purchase", "mean"),
        mean_hist_len=("hist_len", "mean"),
        median_hist_len=("hist_len", "median"),
        mean_slate_len=("slate_len", "mean"),
    )
    print(summary.round(4), "\n")
    # no query may appear in more than one split
    n_splits_per_query = df.groupby("query_key")["split"].nunique()
    assert (n_splits_per_query == 1).all(), "Leak: a query appears in multiple splits"
    print("OK: every query is in exactly one split")
    n_eval = df[df["has_purchase"] == 1].groupby("split").size()
    print("Purchase slates in each group:\n", n_eval.to_string())

In [16]:
df = split_data(user_df)
check_split(df)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val1_df  = df[df["split"] == "val1"].reset_index(drop=True)
val2_df  = df[df["split"] == "val2"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

Rows per split:
 split
train    0.7023
val1     0.1024
test     0.0979
val2     0.0974
Name: proportion, dtype: float64 

         rows  purchase_rate  mean_hist_len  median_hist_len  mean_slate_len
split                                                                       
test    17025         0.1188       152.1524             78.0         88.1472
train  122083         0.1180       153.3412             78.0         89.3528
val1    17799         0.1093       150.6403             76.0         89.3125
val2    16924         0.1140       158.0751             80.0         89.2921 

OK: every query is in exactly one split
Purchase slates in each group:
 split
test      2023
train    14408
val1      1945
val2      1930


In [17]:
train_df.head()

,slate_id,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list,history_qry_list_terms,history_has_query,history_days_before,candidate_has_metadata,history_has_metadata,query_key,has_purchase,hist_len,slate_len,stratum,split
0,0,['63995226' '60257419' '61002446' '25575117'],[97857938 38739023 60065049 34921664 73938759 ...,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0...,['-1' '-1' '-1' '68784030' '36547097\x18602574...,[ 9227588 52788392 4585436 87422901 57364733 ...,['ORD' 'ORD' 'CART' 'CART' 'CART' 'CART' 'CART...,[ 0 0 22967659 82448 432 ...,"[array(['-1'], dtype=object) array(['-1'], dty...",[False False False True True True True Tr...,[350.68440972 350.68440972 84.85502315 83.90...,[ True True True True True True True Tr...,[ True False False True True True True Tr...,63995226 60257419 61002446 25575117,0,182,90,0_2_1,train
1,1,['12549634' '27438884'],[37849772 10338877 33382654 30308565 15104374 ...,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0...,['97556135\x1872694219' '46916741' '-1' '-1' '...,[18256630 46206270 65230659 86878921 32743602 ...,['ORD' 'ORD' 'ORD' 'CART' 'CART' 'CART' 'CART'...,[ 0 4813478 1115463 14061983 920 ...,"[array(['97556135', '72694219'], dtype=object)...",[ True True False False False False False Tr...,[320.8596412 265.14809028 252.23763889 89.48...,[ True True True True True True True Tr...,[ True True True True True True True Tr...,12549634 27438884,0,46,89,0_1_1,train
2,3,['29463948' '63209534' '93568451' '64880503' '...,[14732314 31265387 53977576 90043344 64739295 ...,[0 0 0 0 0 0 0 0 0 0 1 0 0],['34262341\x1851020854\x1859833693\x1813558347...,[49854616 56947170 16552251 91485122 25087051 ...,['CLICK' 'CLICK' 'CLICK' 'CLICK' 'CLICK' 'CLIC...,[ 0 12 4 6 4 14 ...,"[array(['34262341', '51020854', '59833693', '1...",[ True True True True True True True Tr...,[5.04646991e+00 5.04633102e+00 5.04628472e+00 ...,[ True True True True True True True Tr...,[ True True True True True True True Tr...,29463948 63209534 93568451 64880503 93110543 2...,0,205,13,0_3_0,train
3,7,['98384852' '30195534' '32727031'],[79817796 28934316 39809931 82993270 86769848 ...,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0...,['42410833' '42410833' '42410833' '42410833' '...,[ 227329 3396709 33404238 32056400 77953724 ...,['ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'OR...,[ 0 0 0 0 0 ...,"[array(['42410833'], dtype=object) array(['424...",[ True True True True True True True Tr...,[3.42322697e+02 3.42322697e+02 3.42322697e+02 ...,[ True True True True True True True Tr...,[ True True True True True True True Tr...,98384852 30195534 32727031,0,176,112,0_2_3,train
4,8,['14763507' '99551658' '75943140'],[16257946 92361155 64940598 18966247 88670737 ...,[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0...,['37658478\x1896330799\x1863974926\x1896330799...,[13990749 9174292 29422643 9374961 13310147 ...,['ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'ORD' 'OR...,[ 0 446306 0 0 0 712...,"[array(['37658478', '96330799', '63974926', '9...",[ True True True False True False False Fal...,[3.55494606e+02 3.50329028e+02 3.50329028e+02 ...,[ True True True True True True True Tr...,[ True True True True True True False Fal...,14763507 99551658 75943140,0,392,152,0_3_3,train


### Step 1c: Negative sampling
`sample_negatives` takes the TRAIN split only.

For every training slate:
- keep ALL items with label 1/2/3 (positives)
- keep up to `n_neg` (defalut 20) randomly chosen label-0 items (negatives)
- slates with no positives go through the same rule (up to `n_neg` label-0 items)


In [18]:
CAND_COLS = ["candidate_wid_list", "candidate_label_list", "candidate_has_metadata"]

def sample_negatives(train_df, n_neg=20, seed=SEED):
    # Keep all positives (label > 0) + up to n_neg label-0 items per slate.
    # Same columns/rows as train_df; only the candidate lists get shorter.
    rng = np.random.default_rng(seed)
    keeps = []
    for l in train_df["candidate_label_list"].values:
        l = np.asarray(l)
        pos_idx = np.flatnonzero(l > 0)
        neg_idx = np.flatnonzero(l == 0)
        if len(neg_idx) > n_neg:
            neg_idx = rng.choice(neg_idx, size=n_neg, replace=False)
        keeps.append(rng.permutation(np.concatenate([pos_idx, neg_idx])))  # random order inside the slate
    out = train_df.copy()
    for col in CAND_COLS:
        dtype = train_df[col].dtype
        out[col] = pd.Series([np.asarray(x)[k] for x, k in zip(train_df[col], keeps)],
                             index=train_df.index).astype(dtype)
    return out

def check_sampling(train_df, sampled, n_neg=20):
    # sampled has the same rows/columns as train_df; only the candidate lists are shorter
    assert (sampled["slate_id"].values == train_df["slate_id"].values).all(), "rows out of order"
    n_items = sampled["candidate_wid_list"].list.len()
    assert (n_items == sampled["candidate_label_list"].list.len()).all(), "wid/label lengths differ"
    assert (n_items == sampled["candidate_has_metadata"].list.len()).all(), "wid/has_metadata lengths differ"

    print("Slates:", len(sampled), "of", len(train_df))
    print("Candidates before:", train_df["candidate_wid_list"].list.len().sum(),
          "| after:", n_items.sum())
    print("Items per slate (min/25th/median/75th/max):\n", n_items.quantile([0, 0.25, 0.5, 0.75, 1.0]))

    labels = sampled["candidate_label_list"].list.flatten()
    print("Label counts after sampling:\n", labels.value_counts().sort_index().to_string())

    # every positive must survive sampling
    pos_before = (train_df["candidate_label_list"].list.flatten() > 0).sum()
    pos_after = (labels > 0).sum()
    assert pos_before == pos_after, f"positives lost: {pos_before} -> {pos_after}"

    neg_per_slate = sampled["candidate_label_list"].apply(lambda x: int((np.asarray(x) == 0).sum()))
    assert neg_per_slate.max() <= n_neg
    print("OK: all", pos_after, "positives kept; at most", n_neg, "label-0 items per slate")

In [19]:
train_sampled_df = sample_negatives(train_df, n_neg=20)
check_sampling(train_df, train_sampled_df)
print(train_sampled_df.head())

train_df.to_parquet(DATA_DIR / "train.parquet", index=False)
train_sampled_df.to_parquet(DATA_DIR / "train_sampled.parquet", index=False)
val1_df.to_parquet(DATA_DIR / "val1.parquet", index=False)
val2_df.to_parquet(DATA_DIR / "val2.parquet", index=False)
test_df.to_parquet(DATA_DIR / "test.parquet", index=False)

Slates: 122083 of 122083
Candidates before: 10908454 | after: 2608369
Items per slate (min/25th/median/75th/max):
 0.00     1.0
0.25    21.0
0.50    21.0
0.75    22.0
1.00    72.0
dtype: double[pyarrow]
Label counts after sampling:
 0    2382711
1     157802
2      53082
3      14774
OK: all 225658 positives kept; at most 20 label-0 items per slate
   slate_id                                              query  \
0         0      ['63995226' '60257419' '61002446' '25575117']   
1         1                            ['12549634' '27438884']   
2         3  ['29463948' '63209534' '93568451' '64880503' '...   
3         7                 ['98384852' '30195534' '32727031']   
4         8                 ['14763507' '99551658' '75943140']   

                                  candidate_wid_list  \
0  [48645816 60156606   693764 35010767 42508190 ...   
1  [30857980  6115526 85442022 47506004 33213889 ...   
2  [53977576 90043344 83859812 97234676 31265387 ...   
3  [39809931 82948709 730955